> See where is the masks and then extract (87x87) parts from the image

In [ ]:
#| default_exp extract_dissimilar_pins

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
from typing import NewType, List, Tuple, Union, Dict
from pathlib import Path
from scipy.ndimage import label, find_objects
from tqdm.auto import tqdm
from argparse import ArgumentParser, BooleanOptionalAction
import matplotlib as mpl
from fastcore.all import *
from argparse import ArgumentParser, BooleanOptionalAction

In [ ]:
#| export
path=Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection/nbs')
custom_lib_path=Path(r'/home/ai_warstein/custom_libs/goni/custom_libs')
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv
dotenv_p = load_dotenv(Path(Path(path).parent, 'private_easy_pin_detection/.env'))
CV_TOOLS = os.getenv('CV_TOOLS')
PRIVATE_EASY_PIN_DETECTION = os.getenv('PRIVATE_EASY_PIN_DETECTION')
sys.path.append(CV_TOOLS)
sys.path.append(PRIVATE_EASY_PIN_DETECTION)

In [ ]:
import os
os.environ['HF_HOME'] = '/home/ai_warstein/data/huggingface'

In [ ]:
from transformers import AutoModelForMaskGeneration, AutoProcessor, pipeline

In [ ]:
#| export
from cv_tools.core import *
#from labeling_test.grouding_dino_labeling import *

In [ ]:
from labeling_test.grouding_dino_labeling import *

In [ ]:
#mpl.rcParams['image.cmap']='gray'

In [ ]:
im_path = Path(r'/home/ai_sintercra.work/Failed_images_vision_pc/Failed_images_2024_04_23_unzip/main_im2_cropped_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/additional/images')
#im_path = Path(r'/home/ai_sintercra.work/Failed_images_vision_pc/Failed_images_2024_04_23_unzip/main_im1_cropped')
fn=im_path.ls()[0].as_posix()

In [ ]:
m_path = Path(r'/home/ai_sintercra.work/Failed_images_vision_pc/Failed_images_2024_04_23_unzip/main_im2_cropped_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/additional/addition_pin_sn_images')
m_path.ls()

In [ ]:
for i in m_path.ls():
    p_ = i.parent
    name_ = i.name
    new_ne = name_[name_.find('In'):]
    i.rename(f'{p_}/{new_ne}')

In [ ]:
fns = list(filter(lambda x: '2_1'in Path(x).name, m_path.ls()))

In [ ]:
for i in fns:
    p_ = i.parent
    print(p_)
    i.rename(p_/f'w_o_p.png')

In [ ]:
cv2.imwrite(f'{p_}/w_o_p.png', img[:,:-11])

In [ ]:
img = read_img(f'{p_}/w_o_p.png')
show_(img[:, :-9 ])

In [ ]:
[i for i in im_path.ls()]

In [ ]:
from PIL import Image

# Open the image
img = Image.open('your_image_file.jpg')

# Get the size of the image
width, height = img.size

# Calculate the new dimensions for each part
new_width = width // 2
new_height = height // 2

# Crop and save the 4 parts of the image
top_left = img.crop((0, 0, new_width, new_height))
top_left.save('top_left_part.jpg')

top_right = img.crop((new_width, 0, width, new_height))
top_right.save('top_right_part.jpg')

bottom_left = img.crop((0, new_height, new_width, height))
bottom_left.save('bottom_left_part.jpg')

bottom_right = img.crop((new_width, new_height, width, height))
bottom_right.save('bottom_right_part.jpg')

In [ ]:
img = read_img(fn)
cv2.imwrite(
    f'{m_path}/verbogene_pins.png', 
    img[115:190, 1523:1620]
    )

In [ ]:
labels=["circles."]
# refine the mask to get more accurate polygons
polygon_refinement=True
# model threshold , we can adjust this value to get more or less predictions
threshold=0.3
# which model we want to use for object detection
detector_id="IDEA-Research/grounding-dino-tiny"
# which model we want to use for segentation
segmenter_id="facebook/sam-vit-base"    
# just load the image using PIL
img = load_image(fn)
# testing together
cv_img, segs = grounding_dino_segmentation(
    image=img,
    labels=["a cat.", "a remote control."],
    threshold=threshold,
    polygon_refinement=polygon_refinement,
    detector_id=detector_id,
    segmenter_id=segmenter_id)

In [ ]:
annotated_img = get_annotated_img(cv_img, segs)

In [ ]:
show_(annotated_img)

In [ ]:
base_path = Path(r'/home/ai_sintercra.work/Fail_start20240402/_unzip/main_im2_cropped_masks/time_18_17_57_val_frGrnd0.9570_epoch_180.h5/failed/additional')
base_path=Path(r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/additional')
im_path = f'{base_path}/images'
msk_path = f'{base_path}/masks'
fns = Path(msk_path).filter_('97')
fns

In [ ]:
id = 3
sm_msk = read_img(fns[id])
sm_img = read_img(f'{im_path}/{fns[id].stem}.png')
cntrs=find_contours_binary(sm_msk)

In [ ]:
#| export

def frm_cntr_to_bbox(cntr):
    x,y,w,h = cv2.boundingRect(cntr)
    return x,y,w,h

In [ ]:
#| export
def compare_sim_and_save(
        im_path:str,
        msk_path:str,
        offset:int=40,
        save_im_path:str=None,
        save_msk_path:str=None,
        show:bool=False

    ):
    'Compare first pins with others and then saved dissimilar ones'
    if save_im_path: Path(save_im_path).mkdir(exist_ok=True,parents=True)
    if save_msk_path: Path(save_msk_path).mkdir(exist_ok=True,parents=True)

    try:
        img = read_img(im_path)
    except Exception as e:
        print('image could not be read')
        print(e)
    # mask is read only to get position of selected pins
    msk = read_img(msk_path)

    pins, not_pins =  [], []
    cntrs = find_contours_binary(msk)

    cntrs_ = filter(lambda x: cv2.contourArea(x) > 10, cntrs)
    s_cntrs = filter(lambda x: cv2.contourArea(x) <= 10, cntrs)
    for idx, i in enumerate(s_cntrs):
        x, y, w, h = frm_cntr_to_bbox(i)
        pin_img_s = img[y-offset:y+h+offset,x-offset:x+w+offset]
        if save_im_path:
            cv2.imwrite(f'{save_im_path}/{Path(im_path).stem}_s_c_{idx}.png',i[0])
                
        if save_msk_path:
            cv2.imwrite(f'{save_msk_path}/{Path(im_path).stem}_s_c_{idx}.png',i[1])
    for idx , i in enumerate(cntrs_):
        x,y,w,h = frm_cntr_to_bbox(i)
        pin_img_s = img[y-offset:y+h+offset,x-offset:x+w+offset]
        pin_img = img[y:y+h,x:x+w]
        pin_msk = msk[y:y+h,x:x+w]
        if idx == 0:
            ref_img = pin_img

        else:
            pin_img_r = cv2.resize(
                pin_img,
                (ref_img.shape[1],ref_img.shape[0])
            )
            sim = ssim_(
                ref_img,
                pin_img_r,
                win_size=None,
            )
            if sim < 0.45:
                not_pins.append([pin_img_s, pin_msk])
            else:
                pins.append([pin_img_s, pin_msk])
    
    # there is a possibility my reffernce pin is not actual pin
    # then similarity will be low for all pins , but here i am 
    # checking which has bigger length, pins will be always bigger length
    if (len(pins) > len(not_pins)) and len(not_pins) > 0:
        for idx,i in enumerate(not_pins):
            if show: show_(i)
            if save_im_path:
                cv2.imwrite(f'{save_im_path}/{Path(im_path).stem}_{idx}.png',i[0])
                
            if save_msk_path:
                cv2.imwrite(f'{save_msk_path}/{Path(im_path).stem}_{idx}.png',i[1])
        
    elif (len(pins)< len(not_pins)) and len(pins) > 0:
        for i in pins:
            if show: show_(i)
            if save_im_path:
                cv2.imwrite(f'{save_im_path}/{Path(im_path).stem}_{idx}.png',i[0])
            if save_msk_path:
                cv2.imwrite(f'{save_msk_path}/{Path(im_path).stem}_{idx}.png',i[1])

In [ ]:
#| export
def compare_sim_and_save_folder(
        im_path:str, #Could be image list or path to folder
        msk_path:str,#where the masks are saved, will be searched with same name as image
        filter_w:str=None, # whether to filter for specififc words in the files
        offset:int=40,# how much extra area to consider around pin
        save_im_path:str=None, # where to save the images
        save_msk_path:str=None, # where to save the masks
        show:bool=False # whether to show the images or not

    ):

    if filter_w is not None:
        im_list = Path(im_path).filter_(filter_w)
    else:
        im_list = Path(im_path).ls()

    for i in tqdm(im_list, total=len(im_list)):
        name_=Path(i).name
        compare_sim_and_save(
            im_path=i,
            msk_path=f'{msk_path}/{name_}',
            offset=offset,
            save_im_path=save_im_path,
            save_msk_path=save_msk_path,
        )


In [ ]:
#im_name='VFV4.7.9.5_2024040501484650_ID_00568045421818262692412_In_97_r_1_FRONT_Missing Lead_image2.png'
#im_path=f'{base_path}/images/{im_name}'
#img = read_img(im_path)
#Path(base_path, 'images').ls()

In [ ]:
for idx, i in enumerate(im_path.ls(),total=len(im-path.ls())):
    

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        '--im_path',
        type=str,
        help='Path to images'
    )
    parser.add_argument(
        '--msk_path',
        type=str,
        help='Path to masks'
    )
    parser.add_argument(
        '--filter_w',
        type=str,
        help='Filter for specific words in the files'
    )
    parser.add_argument(
        '--offset',
        type=int,
        default=40,
        help='How much extra area to consider around pin'
    )
    parser.add_argument(
        '--save_im_path',
        default=None,
        help='Where to save the images'
    )
    parser.add_argument(
        '--save_msk_path',
        default=None,
        help='Where to save the masks'
    )
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_()
    compare_sim_and_save_folder(
        im_path=args.im_path,
        msk_path=args.msk_path,
        filter_w=args.filter_w,
        offset=args.offset,
        save_im_path=args.save_im_path,
        save_msk_path=args.save_msk_path,
    )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('22_extract_pins_from_mask.ipynb')